In [1]:
import torch
import polars as pl

In [2]:
# loading the dataset from hugging face
from datasets import load_dataset
dataset = load_dataset("mohamed-khalil/ATHAR")

In [3]:
#separating the data into training and testing data
train_data=dataset['train'].to_polars()
test_data=dataset['test'].to_polars()

In [4]:
def tokenize_native(t):
  # /s to match any whitespace
  t=t.with_columns(
    pl.all()
    .str.to_lowercase() # convert the english letters to lower case
    .str.replace_all(r'http\S+|www\S+|@|#', '') # strip out the links and special characters
    .str.replace_all(r'[^\w\s]', ' ') # delete any non alphanumeric word followed by a space
    .str.replace_all(r'\s+', ' ') # delete any consecutive spaces
    .str.strip_chars()
  ).with_columns(
    pl.format('<eos> {} <sos>', pl.col('arabic')), # add beginning and ending of the sentence from right to left
    pl.format('<sos {} <sos>', pl.col('english')) # add beginning and ending of the sentence from left to right
  ).with_columns(
      pl.all().str.split(' ') # tokenize words
  )
  return t

train_data=tokenize_native(train_data)
test_data=tokenize_native(test_data)

In [5]:
# Pad polars list to the maximum one
# find the maximum list
def padding_df(t):
# add padding to the data to ensure they are compatible to be converted to a tensor type
  for col in t.columns:
    max=t.select(pl.col(col)).with_columns(pl.col(col).list.len()).max().item(0,0) # take the maximum length of the list
    t=t.with_columns(
        pl.col(col).list.concat(
            pl.lit("<pad>").repeat_by(max-pl.col(col).list.len()) # to ensure compatibleness, repeat by whole size - list len
            # for example if the list len is 3 max is 8 then add only five elements
        ).cast(pl.List(pl.Categorical)).to_physical()) # encode the categorical data to be accepted in torch

  return t

train_data=padding_df(train_data)
test_data=padding_df(test_data)


In [6]:
def cast_Utf(t):
  for col in t.columns:
    max=t.select(pl.col(col)
    ).with_columns(
        pl.col(col).list.len()
        ).max().item(0,0) # take the maximum length of the list
    t=t.with_columns(
      pl.col(col).cast(pl.Array(pl.UInt32, shape=(max)))
    )

  return t

train_data=cast_Utf(train_data)
test_data=cast_Utf(test_data)

In [7]:
train_data

arabic,english
"array[u32, 185]","array[u32, 203]"
"[0, 1, … 25]","[0, 1, … 32]"
"[0, 26, … 25]","[0, 33, … 32]"
"[0, 59, … 25]","[0, 67, … 32]"
"[0, 66, … 25]","[0, 33, … 32]"
"[0, 123, … 25]","[0, 113, … 32]"
…,…
"[0, 108365, … 25]","[0, 95, … 32]"
"[0, 189615, … 25]","[0, 1831, … 32]"
"[0, 189525, … 25]","[0, 33, … 32]"


In [ ]:
##TODO: prepare the training engine
#TODO: prepare the inference engine
# code for inference to get back the data to the original format
train_data.with_columns(pl.all().cast(pl.List(pl.Categorical)))

## Data Analysis and Preprocessing
- in this part you are required to to conduct proper analysis of the above data
- You are also required to preprocess the above data in manner where is ready for modelling

## Modelling Section
- In this part you are required to build two models transformer and Attention based sequence to sequence model.

In [9]:
import torch
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val= train_test_split(train_data['arabic'], train_data['english'], test_size=0.3)
x_train, x_val, y_train, y_val=x_train.to_torch(),  x_val.to_torch(), y_train.to_torch(), y_val.to_torch()


ModuleNotFoundError: No module named 'sklearn'